# Paper 2 — MTJ Long-Term/Short-Term Stochastic Synapse with On-Chip STDP
**Srinivasan, Sengupta & Roy (2016), Scientific Reports 6, 29545.** [DOI](https://doi.org/10.1038/srep29545)

This runnable compact-model reconstruction covers the three-terminal MTJ–heavy-metal synapse, probabilistic binary switching, timing-dependent STDP, paired LT/ST elements, read/write paths, and a small network demonstration. It is not an exact reproduction of the paper's MNIST accuracy or transistor-level SPICE figures. Defaults are illustrative; use paper-table values for quantitative matching.


## 1. Device transmission model

\[
I_{post}=V_{pre}G(s),\quad
G(s)=G_{AP}+\frac{s+1}{2}(G_P-G_{AP}),\quad s\in\{-1,+1\}.
\]
The HM programming path is functionally separate from the MTJ read/transmission path.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
@dataclass
class MTJ:
    Gp: float=20e-6
    Gap: float=5e-6
    def conductance(self,s): return self.Gap+(np.asarray(s)+1)/2*(self.Gp-self.Gap)
    def transmit(self,v,s): return np.asarray(v)*self.conductance(s)
mtj=MTJ()
print("G(AP), G(P) in µS:",mtj.conductance([-1,1])*1e6)


## 2. Exponential stochastic STDP

For \(\Delta t=t_{post}-t_{pre}\):
\[
p_{pot}=p_{max,+}e^{-\Delta t/\tau_+}\;(\Delta t>0),\qquad
p_{dep}=p_{max,-}e^{\Delta t/\tau_-}\;(\Delta t<0).
\]
The paper reports a peak probability around 0.15 in its illustrated setting. Time constants and peak probabilities are exposed below.

In [ ]:
pmax_p=pmax_d=0.15
tau_p=tau_d=2.0 # µs; illustrative
def stdp_prob(dt):
    dt=np.asarray(dt); p=np.zeros_like(dt,dtype=float)
    pos=dt>0; neg=dt<0
    p[pos]=pmax_p*np.exp(-dt[pos]/tau_p)
    p[neg]=pmax_d*np.exp(dt[neg]/tau_d)
    return p
d=np.linspace(-10,10,501)
plt.figure(figsize=(7,4)); plt.plot(d,stdp_prob(d)); plt.xlabel('Δt (µs)'); plt.ylabel('switch probability'); plt.grid(); plt.show()


## 3. Binary state update and Monte Carlo validation

In [ ]:
def update_binary(state,dt,rng):
    if rng.random()<float(stdp_prob(np.array([dt]))[0]):
        if dt>0: return 1
        if dt<0: return -1
    return int(state)
rng=np.random.default_rng(7); N=20000
for delay in [.5,2,6,-.5,-2,-6]:
    s=-1 if delay>0 else 1
    changed=sum(update_binary(s,delay,rng)!=s for _ in range(N))/N
    print(delay, "µs predicted",float(stdp_prob(np.array([delay]))[0]),"observed",changed)


## 4. Paired long-term/short-term synapse

LT favors tightly correlated events; ST responds over a broader timing range and forgets faster. The following is a transparent compact abstraction of those design principles, not a verbatim transcription of every transition detail in the paper.

In [ ]:
class LTST:
    def __init__(self,seed=3):
        self.lt=self.st=-1; self.rng=np.random.default_rng(seed)
        self.lt_tau,self.st_tau=1.,4.
        self.lt_peak=self.st_peak=.15
        self.lt_forget,self.st_forget=.001,.02
    def step(self,dt):
        for name,tau,peak,forget in [('lt',self.lt_tau,self.lt_peak,self.lt_forget),('st',self.st_tau,self.st_peak,self.st_forget)]:
            x=getattr(self,name)
            if self.rng.random()<peak*np.exp(-abs(dt)/tau): x=1 if dt>0 else -1
            if x==1 and self.rng.random()<forget: x=-1
            setattr(self,name,x)
    def output(self,v=1.):
        return .5*(mtj.transmit(v,self.lt)+mtj.transmit(v,self.st))
syn=LTST(); tr=[]
for k in range(5000):
    syn.step(.7 if k%3 else -2.5); tr.append([syn.lt,syn.st,syn.output()*1e6])
tr=np.array(tr)
fig,ax=plt.subplots(3,1,figsize=(9,6),sharex=True)
for i,label in enumerate(['LT state','ST state','output (µA at 1 V)']): ax[i].plot(tr[:,i],lw=.7); ax[i].set_ylabel(label)
ax[-1].set_xlabel('event index'); plt.tight_layout(); plt.show()


## 5. Forgetting/retention sweep

In [ ]:
def retention(f,n=2000,trials=1000,seed=1):
    r=np.random.default_rng(seed); alive=np.ones(trials,bool); out=[]
    for _ in range(n):
        alive &= ~(r.random(trials)<f); out.append(alive.mean())
    return np.arange(n),np.array(out)
plt.figure(figsize=(7,4))
for f in [.0005,.002,.01,.03]:
    x,y=retention(f); plt.plot(x,y,label=f'forget={f}')
plt.xlabel('event count'); plt.ylabel('fraction retained'); plt.legend(); plt.grid(); plt.show()


## 6. Equivalent circuit and energy estimates

\[
I_{post}=V_{pre}G(s),\quad E_{write}=I_{HM}^2R_{HM}t_p,\quad
E_{read}=V_{read}^2G(s)t_{read}.
\]
These first-order Joule estimates omit peripheral and driver energy.

In [ ]:
def energy(I,R,t): return I**2*R*t
Ew=energy(100e-6,500.,1e-9)
Er=.1**2*mtj.Gp*1e-9
print(f"Illustrative write={Ew/1e-15:.3f} fJ; read={Er/1e-15:.3f} fJ")


## 7. Tiny network demonstration (not the paper's MNIST result)

In [ ]:
rng=np.random.default_rng(11); nin,nout=64,4
W=rng.choice([-1,1],(nin,nout))
X=np.zeros((4,nin))
for c in range(4): X[c,c*16:(c+1)*16]=1
G=mtj.conductance(W)
scores=X@G
print('Scores:',scores, '\nPredicted class:',scores.argmax(1))


## 8. Reproduction checklist and next steps

- Transcribe exact device and circuit parameters from the paper's tables.
- Digitize and fit switching-probability/STDP curves.
- Match the precise LT/ST significance and state-transition logic.
- Rebuild network topology, neuron settings, MNIST preprocessing and training schedule before comparing accuracy.
- Compare convergence and iso-accuracy programming energy.
- For higher fidelity, calibrate switching from stochastic LLG/micromagnetics and export read/write paths to SPICE.

Suggested open tools: Brian2 (SNN), NumPy/SciPy (compact model), ngspice/PySpice (circuit), MuMax3 or OOMMF/Ubermag (micromagnetics).
